# Fake News Detection — Retraining on Cleaned ISOT (Reuters Tag Stripped)


## 1. Load Original Splits and Strip Dateline


In [1]:
import pandas as pd
import re

train_df = pd.read_csv("train.csv")
val_df = pd.read_csv("val.csv")
test_df = pd.read_csv("test.csv")

def strip_dateline(text, check_chars=150):
    """Removes a leading wire-service dateline pattern, e.g. 'WASHINGTON (Reuters) - '."""
    pattern = r"^.{0,80}\(Reuters\)\s*[-–—]\s*"
    stripped = re.sub(pattern, "", str(text)[:check_chars], count=1) + str(text)[check_chars:]
    return stripped

for df in [train_df, val_df, test_df]:
    df["text_original"] = df["text"]
    df["text"] = df["text"].apply(strip_dateline)

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

# Sanity check: confirm stripping worked
def has_reuters_dateline(text, check_chars=100):
    return bool(re.search(r"\(Reuters\)", str(text)[:check_chars]))

real_train = train_df[train_df["label"] == "real"]
print(f"\nReal articles in train still containing dateline after stripping: {real_train['text'].apply(has_reuters_dateline).mean():.2%}")


Train: (27044, 9)
Val: (5795, 9)
Test: (5796, 9)

Real articles in train still containing dateline after stripping: 0.01%


## 2. Retrain Baseline: TF-IDF + Logistic Regression (Cleaned Data)


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
import time

baseline_clean = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=20000,
        ngram_range=(1, 2),
        stop_words="english",
        min_df=2
    )),
    ("clf", LogisticRegression(max_iter=1000, random_state=42))
])

start = time.time()
baseline_clean.fit(train_df["text"], train_df["label_id"])
train_time = time.time() - start

val_pred = baseline_clean.predict(val_df["text"])
val_acc = accuracy_score(val_df["label_id"], val_pred)
val_prec, val_rec, val_f1, _ = precision_recall_fscore_support(val_df["label_id"], val_pred, average="binary")
print(f"Validation Accuracy: {val_acc:.4f}")

start = time.time()
test_pred = baseline_clean.predict(test_df["text"])
infer_time = time.time() - start
test_acc = accuracy_score(test_df["label_id"], test_pred)
test_prec, test_rec, test_f1, _ = precision_recall_fscore_support(test_df["label_id"], test_pred, average="binary")

print(f"\nTest Accuracy: {test_acc:.4f}")
print(classification_report(test_df["label_id"], test_pred, target_names=["real", "fake"]))

baseline_clean_results = {
    "model": "TF-IDF + Logistic Regression (cleaned)",
    "val_accuracy": val_acc, "val_precision": val_prec, "val_recall": val_rec, "val_f1": val_f1,
    "test_accuracy": test_acc, "test_precision": test_prec, "test_recall": test_rec, "test_f1": test_f1,
    "train_time_sec": train_time, "inference_time_sec": infer_time,
}

import joblib
joblib.dump(baseline_clean, "baseline_pipeline_cleaned.joblib")
baseline_clean_results


Validation Accuracy: 0.9796



Test Accuracy: 0.9791
              precision    recall  f1-score   support

        real       0.97      0.99      0.98      3179
        fake       0.99      0.97      0.98      2617

    accuracy                           0.98      5796
   macro avg       0.98      0.98      0.98      5796
weighted avg       0.98      0.98      0.98      5796



{'model': 'TF-IDF + Logistic Regression (cleaned)',
 'val_accuracy': 0.9796376186367558,
 'val_precision': 0.9871294851794071,
 'val_recall': 0.9675076452599388,
 'val_f1': 0.9772200772200772,
 'test_accuracy': 0.9791235334713596,
 'test_precision': 0.985981308411215,
 'test_recall': 0.9675200611387085,
 'test_f1': 0.9766634522661524,
 'train_time_sec': 25.455848217010498,
 'inference_time_sec': 2.277498245239258}

## 3. Retrain DistilBERT (Cleaned Data)

Same hyperparameters as the original ISOT run (learning rate 2e-5, batch size 16, up to 5 epochs, early stopping patience=2, max length 256), so the comparison is fair — only the data changed.


In [3]:
import torch
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 256

tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

def to_hf_dataset(df):
    return Dataset.from_pandas(
        df[["text", "label_id"]].rename(columns={"label_id": "labels"}).reset_index(drop=True)
    )

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=MAX_LENGTH)

train_ds = to_hf_dataset(train_df).map(tokenize_fn, batched=True)
val_ds = to_hf_dataset(val_df).map(tokenize_fn, batched=True)
test_ds = to_hf_dataset(test_df).map(tokenize_fn, batched=True)

for ds in [train_ds, val_ds, test_ds]:
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1}

model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)

training_args = TrainingArguments(
    output_dir="./distilbert_isot_cleaned",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model, args=training_args, train_dataset=train_ds, eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

start_train = time.time()
trainer.train()
distilbert_clean_train_time = time.time() - start_train
print(f"Training time: {distilbert_clean_train_time:.2f}s")


Using device: cuda


Map:   0%|          | 0/27044 [00:00<?, ? examples/s]

Map:   0%|          | 0/5795 [00:00<?, ? examples/s]

Map:   0%|          | 0/5796 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training time: 2076.30s


In [4]:
start_infer = time.time()
test_output = trainer.predict(test_ds)
distilbert_clean_infer_time = time.time() - start_infer

test_preds = test_output.predictions.argmax(axis=-1)
test_labels = test_output.label_ids

distilbert_clean_acc = accuracy_score(test_labels, test_preds)
distilbert_clean_prec, distilbert_clean_rec, distilbert_clean_f1, _ = precision_recall_fscore_support(
    test_labels, test_preds, average="binary"
)

print(f"Test Accuracy: {distilbert_clean_acc:.4f}")
print(classification_report(test_labels, test_preds, target_names=["real", "fake"]))

distilbert_clean_results = {
    "model": "DistilBERT (cleaned)",
    "test_accuracy": distilbert_clean_acc, "test_precision": distilbert_clean_prec,
    "test_recall": distilbert_clean_rec, "test_f1": distilbert_clean_f1,
    "train_time_sec": distilbert_clean_train_time, "inference_time_sec": distilbert_clean_infer_time,
}

model.save_pretrained("./distilbert_isot_cleaned_final")
tokenizer.save_pretrained("./distilbert_isot_cleaned_final")
distilbert_clean_results


Test Accuracy: 0.9972
              precision    recall  f1-score   support

        real       1.00      1.00      1.00      3179
        fake       1.00      1.00      1.00      2617

    accuracy                           1.00      5796
   macro avg       1.00      1.00      1.00      5796
weighted avg       1.00      1.00      1.00      5796



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'model': 'DistilBERT (cleaned)',
 'test_accuracy': 0.9972394755003451,
 'test_precision': 0.9969430645777608,
 'test_recall': 0.9969430645777608,
 'test_f1': 0.9969430645777608,
 'train_time_sec': 2076.2975413799286,
 'inference_time_sec': 30.144774436950684}

## 4. Retrain RoBERTa (Cleaned Data)


In [5]:
from transformers import RobertaTokenizerFast, RobertaForSequenceClassification

ROBERTA_MODEL_NAME = "roberta-base"
roberta_tokenizer = RobertaTokenizerFast.from_pretrained(ROBERTA_MODEL_NAME)

def roberta_tokenize_fn(batch):
    return roberta_tokenizer(batch["text"], truncation=True, padding="max_length", max_length=MAX_LENGTH)

roberta_train_ds = to_hf_dataset(train_df).map(roberta_tokenize_fn, batched=True)
roberta_val_ds = to_hf_dataset(val_df).map(roberta_tokenize_fn, batched=True)
roberta_test_ds = to_hf_dataset(test_df).map(roberta_tokenize_fn, batched=True)

for ds in [roberta_train_ds, roberta_val_ds, roberta_test_ds]:
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

roberta_model = RobertaForSequenceClassification.from_pretrained(ROBERTA_MODEL_NAME, num_labels=2)
roberta_model.to(device)

roberta_training_args = TrainingArguments(
    output_dir="./roberta_isot_cleaned",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    report_to="none"
)

roberta_trainer = Trainer(
    model=roberta_model, args=roberta_training_args,
    train_dataset=roberta_train_ds, eval_dataset=roberta_val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

start_train = time.time()
roberta_trainer.train()
roberta_clean_train_time = time.time() - start_train
print(f"Training time: {roberta_clean_train_time:.2f}s")


Map:   0%|          | 0/27044 [00:00<?, ? examples/s]

Map:   0%|          | 0/5795 [00:00<?, ? examples/s]

Map:   0%|          | 0/5796 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training time: 4047.04s


In [6]:
start_infer = time.time()
roberta_test_output = roberta_trainer.predict(roberta_test_ds)
roberta_clean_infer_time = time.time() - start_infer

roberta_test_preds = roberta_test_output.predictions.argmax(axis=-1)
roberta_test_labels = roberta_test_output.label_ids

roberta_clean_acc = accuracy_score(roberta_test_labels, roberta_test_preds)
roberta_clean_prec, roberta_clean_rec, roberta_clean_f1, _ = precision_recall_fscore_support(
    roberta_test_labels, roberta_test_preds, average="binary"
)

print(f"Test Accuracy: {roberta_clean_acc:.4f}")
print(classification_report(roberta_test_labels, roberta_test_preds, target_names=["real", "fake"]))

roberta_clean_results = {
    "model": "RoBERTa (cleaned)",
    "test_accuracy": roberta_clean_acc, "test_precision": roberta_clean_prec,
    "test_recall": roberta_clean_rec, "test_f1": roberta_clean_f1,
    "train_time_sec": roberta_clean_train_time, "inference_time_sec": roberta_clean_infer_time,
}

roberta_model.save_pretrained("./roberta_isot_cleaned_final")
roberta_tokenizer.save_pretrained("./roberta_isot_cleaned_final")
roberta_clean_results


Test Accuracy: 0.9645
              precision    recall  f1-score   support

        real       0.96      0.97      0.97      3179
        fake       0.97      0.95      0.96      2617

    accuracy                           0.96      5796
   macro avg       0.96      0.96      0.96      5796
weighted avg       0.96      0.96      0.96      5796



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'model': 'RoBERTa (cleaned)',
 'test_accuracy': 0.9644582470669427,
 'test_precision': 0.967429236138038,
 'test_recall': 0.9533817348108521,
 'test_f1': 0.9603541185527329,
 'train_time_sec': 4047.036796092987,
 'inference_time_sec': 57.48863363265991}

## 5. Full Comparison: Original vs Cleaned vs LIAR


In [7]:
original_comparison = pd.read_csv("full_model_comparison.csv")
original_acc = dict(zip(original_comparison["model"], original_comparison["test_accuracy"]))

liar_comparison = pd.read_csv("liar_model_comparison.csv")
liar_acc = dict(zip(liar_comparison["model"].replace({
    "TF-IDF + Logistic Regression": "TF-IDF + Logistic Regression",
    "DistilBERT": "DistilBERT",
    "RoBERTa": "RoBERTa"
}), liar_comparison["test_accuracy"]))

showcase = pd.DataFrame([
    {
        "model": "DistilBERT",
        "original_isot_accuracy": original_acc.get("DistilBERT"),
        "cleaned_isot_accuracy": distilbert_clean_results["test_accuracy"],
        "liar_accuracy": liar_acc.get("DistilBERT"),
    },
    {
        "model": "RoBERTa",
        "original_isot_accuracy": original_acc.get("RoBERTa"),
        "cleaned_isot_accuracy": roberta_clean_results["test_accuracy"],
        "liar_accuracy": liar_acc.get("RoBERTa"),
    },
    {
        "model": "TF-IDF + Logistic Regression",
        "original_isot_accuracy": original_acc.get("TF-IDF + Logistic Regression"),
        "cleaned_isot_accuracy": baseline_clean_results["test_accuracy"],
        "liar_accuracy": liar_acc.get("TF-IDF + Logistic Regression"),
    },
])

showcase.to_csv("showcase_comparison_original_cleaned_liar.csv", index=False)
showcase


,model,original_isot_accuracy,cleaned_isot_accuracy,liar_accuracy
0,DistilBERT,0.999655,0.997239,0.682278
1,RoBERTa,0.998792,0.964458,0.607595
2,TF-IDF + Logistic Regression,0.986025,0.979124,0.624051
